In [ ]:
# ======================================================
# Import necessary Libraries
# ======================================================
import tensorflow as tf
import numpy as np
import time
import matplotlib.pyplot as plt
from tensorflow.keras.applications import (
    DenseNet121, EfficientNetB0, EfficientNetB3, EfficientNetV2B0,
    InceptionV3, MobileNet, MobileNetV2, ResNet50V2, VGG16, Xception
)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# ======================================================
# Load the CIFAR-100 Dataset
# ======================================================
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.cifar100.load_data(label_mode='fine')

selected_classes = list(range(20))  # Select first 20 fine classes
num_classes = len(selected_classes)

# ======================================================
# Select 20 classes
# ======================================================
train_mask = np.isin(y_train_full, selected_classes).flatten()
x_train = x_train_full[train_mask]
y_train = y_train_full[train_mask]

test_mask = np.isin(y_test_full, selected_classes).flatten()
x_test = x_test_full[test_mask]
y_test = y_test_full[test_mask]

# ======================================================
# Remap labels in the dataset
# ======================================================
label_mapping = {original: new for new, original in enumerate(selected_classes)}
y_train = np.vectorize(label_mapping.get)(y_train)
y_test = np.vectorize(label_mapping.get)(y_test)

# ======================================================
# Label Encoding Phase
# ======================================================
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

# Convert to TensorFlow Datasets
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train_cat))
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test_cat))

# ======================================================
# Image Preprocessing Phase
# ======================================================
models = [
    ('DenseNet121', DenseNet121),
    ('EfficientNetB0', EfficientNetB0),
    ('EfficientNetB3', EfficientNetB3),
    ('EfficientNetV2B0', EfficientNetV2B0),
    ('InceptionV3', InceptionV3),
    ('MobileNet', MobileNet),
    ('MobileNetV2', MobileNetV2),
    ('ResNet50V2', ResNet50V2),
    ('VGG16', VGG16),
    ('Xception', Xception),
]

def preprocess_image(image, label, model_name, training=False):
    if model_name in ['VGG16', 'Xception']:
        image = tf.image.resize(image, (224, 224))
    else:
        image = tf.image.resize(image, (128, 128))

    if training:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, max_delta=0.1)
        image = tf.image.random_contrast(image, lower=0.9, upper=1.1)

    # Apply model-specific preprocessing
    if model_name == 'DenseNet121':
        image = tf.keras.applications.densenet.preprocess_input(image)
    elif model_name in ['EfficientNetB0', 'EfficientNetB3']:
        image = tf.keras.applications.efficientnet.preprocess_input(image)
    elif model_name == 'EfficientNetV2B0':
        image = tf.keras.applications.efficientnet_v2.preprocess_input(image)
    elif model_name == 'InceptionV3':
        image = tf.keras.applications.inception_v3.preprocess_input(image)
    elif model_name == 'MobileNet':
        image = tf.keras.applications.mobilenet.preprocess_input(image)
    elif model_name == 'MobileNetV2':
        image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    elif model_name == 'ResNet50V2':
        image = tf.keras.applications.resnet_v2.preprocess_input(image)
    elif model_name == 'VGG16':
        image = tf.keras.applications.vgg16.preprocess_input(image)
    elif model_name == 'Xception':
        image = tf.keras.applications.xception.preprocess_input(image)

    return image, label

# ======================================================
# Activation Function Benchmarking Phase
# ======================================================
activation_functions = ["sigmoid", "relu", "softmax", "tanh"]
results = []
start_time = time.time()

for model_name, ModelClass in models:
    for activation_fn in activation_functions:
        print(f"\nTraining {model_name} with {activation_fn} activation...")

        train_ds = train_dataset.map(lambda image, label: preprocess_image(image, label, model_name, training=True))
        train_ds = train_ds.shuffle(buffer_size=1000).batch(batch_size=32).prefetch(buffer_size=1)
        test_ds = test_dataset.map(lambda image, label: preprocess_image(image, label, model_name, training=False))
        test_ds = test_ds.batch(batch_size=32).prefetch(buffer_size=1)

        input_shape = (224, 224, 3) if model_name in ['VGG16', 'Xception'] else (128, 128, 3)
        base_model = ModelClass(weights='imagenet', include_top=False, input_shape=input_shape)
        base_model.trainable = False

        x = base_model.output
        x = GlobalAveragePooling2D()(x)
        x = Dense(128, activation='relu')(x)
        out = Dense(num_classes, activation=activation_fn, dtype='float32')(x)

        model = Model(inputs=base_model.input, outputs=out)
        model.compile(optimizer=Adam(learning_rate=0.005), loss='categorical_crossentropy', metrics=['accuracy'])

        model.fit(train_ds, epochs=5, validation_data=test_ds, verbose=1)
        test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
        results.append((model_name, activation_fn, test_accuracy, test_loss))

end_time = time.time()
print(f"\nTotal benchmarking time: {(end_time - start_time)/60:.2f} minutes")

# Print final results
for r in results:
    print(f"Model: {r[0]:15s} | Activation: {r[1]:8s} | Accuracy: {r[2]*100:.2f}% | Loss: {r[3]:.4f}")



Training DenseNet121 with sigmoid activation...
Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 83s 166ms/step - accuracy: 0.6686 - loss: 1.1563 - val_accuracy: 0.8110 - val_loss: 0.6010
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 50ms/step - accuracy: 0.8365 - loss: 0.4793 - val_accuracy: 0.8290 - val_loss: 0.5293
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 51ms/step - accuracy: 0.8629 - loss: 0.4104 - val_accuracy: 0.8215 - val_loss: 0.6440
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 16s 48ms/step - accuracy: 0.8750 - loss: 0.3597 - val_accuracy: 0.8305 - val_loss: 0.5933
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 50ms/step - accuracy: 0.8838 - loss: 0.3276 - val_accuracy: 0.8265 - val_loss: 0.6283

Training DenseNet121 with relu activation...
Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 76s 159ms/step - accuracy: 0.2230 - loss: nan - val_accuracy: 0.3005 - val_loss: 6.5798
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 51ms/step - accuracy: 0.3550 - loss: nan - val_accuracy: 0.5005 - val_loss: nan
Epoch 3/5